In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (full tables stored in variables)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to tz-aware datetime (UTC -> Asia/Shanghai)
def to_local_dt_series(epoch_series):
    # handle possible NaNs gracefully
    s = pd.to_datetime(epoch_series.dropna().astype(int), unit='s', utc=True)
    s = s.dt.tz_convert(tz)
    # reindex to original index, fill NaT for dropped NaNs
    return s.reindex(epoch_series.index)

# Add local datetime columns (kept in variables but not to display full tables)
df_metric['ts_local'] = to_local_dt_series(df_metric['timestamp'])
df_trace['ts_local'] = to_local_dt_series(df_trace['timestamp'])
df_log['ts_local'] = to_local_dt_series(df_log['timestamp'])
df_err['ts_local'] = to_local_dt_series(df_err['timestamp'])

# 1) File-level summaries: total row count, min timestamp, max timestamp (converted to Asia/Shanghai)
def file_summary_df(df, name):
    total = int(df.shape[0])
    if total == 0:
        min_dt = max_dt = pd.NaT
    else:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
        min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz)
        max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz)
    return pd.DataFrame([{
        'file': name,
        'rows': total,
        'min_timestamp_local': min_dt,
        'max_timestamp_local': max_dt
    }])

files_summary = pd.concat([
    file_summary_df(df_metric, 'metric.csv'),
    file_summary_df(df_trace, 'trace.csv'),
    file_summary_df(df_log, 'log.csv'),
    file_summary_df(df_err, 'error_logs.csv')
], ignore_index=True)

# 2) Aggregations
# Metric: group by (cmdb_id, kpi_name) -> count and global P95 of value
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='count', global_P95_of_value=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.notna().any() else np.nan)
    .reset_index()
)
metric_agg_head = metric_agg.head(50)

# Trace: group by (cmdb_id, trace_name)
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count='count', global_P95_of_value=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.notna().any() else np.nan)
    .reset_index()
)
trace_agg_head = trace_agg.head(50)

# Log: group by (cmdb_id, log_name)
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count='count', global_P95_of_value=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.notna().any() else np.nan)
    .reset_index()
)
log_agg_head = log_agg.head(50)

# 3) error_logs distinct (cmdb_id, message) with counts (up to 20 rows) and earliest 5 timestamps for that file
err_pairs = (
    df_err
    .groupby(['cmdb_id', 'message'], dropna=False)
    .agg(count=('message', 'count'))
    .reset_index()
    .sort_values(['count', 'cmdb_id'], ascending=[False, True])
)
err_pairs_head = err_pairs.head(20)

# earliest 5 timestamps (local) for error_logs.csv
if df_err.shape[0] > 0:
    err_earliest5 = (
        pd.to_datetime(df_err['timestamp'].astype(int), unit='s', utc=True)
        .dt.tz_convert(tz)
        .sort_values()
        .drop_duplicates()
        .head(5)
        .reset_index(drop=True)
    )
else:
    err_earliest5 = pd.Series(dtype='datetime64[ns, Asia/Shanghai]')

# Keep full tables in variables (already stored): df_metric, df_trace, df_log, df_err
# Display compact outputs (variables rather than prints). The tuple will show them in order.
files_summary, metric_agg_head, trace_agg_head, log_agg_head, err_pairs_head, err_earliest5

```
Out[1]:
```
# Summarize the previously computed aggregates in plain English as a compact string.
# Use existing variables from the kernel: files_summary, metric_agg_head, trace_agg_head, log_agg_head, err_pairs_head, err_earliest5

def row_to_line_metric(row):
    return f"- {row['cmdb_id']} | {row['kpi_name']} | count={int(row['count'])} | P95={row['global_P95_of_value']:.6g}"

def row_to_line_trace(row):
    return f"- {row['cmdb_id']} | {row['trace_name']} | count={int(row['count'])} | P95={row['global_P95_of_value']:.6g}"

def row_to_line_log(row):
    return f"- {row['cmdb_id']} | {row['log_name']} | count={int(row['count'])} | P95={row['global_P95_of_value']:.6g}"

lines = []
lines.append("File summaries (rows, min_local_ts, max_local_ts):")
for _, r in files_summary.iterrows():
    lines.append(f"- {r['file']}: rows={int(r['rows'])}, min={r['min_timestamp_local']}, max={r['max_timestamp_local']}")

lines.append("")    
lines.append("Top metric groups (showing up to 3 of the computed top 50):")
for _, r in metric_agg_head.head(3).iterrows():
    lines.append(row_to_line_metric(r))

lines.append("")
lines.append("Top trace groups (showing up to 3 of the computed top 50):")
for _, r in trace_agg_head.head(3).iterrows():
    lines.append(row_to_line_trace(r))

lines.append("")
lines.append("Top log groups (showing up to 3 of the computed top 50):")
for _, r in log_agg_head.head(3).iterrows():
    lines.append(row_to_line_log(r))

lines.append("")
lines.append("Error logs summary:")
if err_pairs_head.shape[0] > 0:
    lines.append(f"- Distinct (cmdb_id, message) pairs returned: {min(20, err_pairs_head.shape[0])}")
    lines.append("- Top error message pairs (up to 3):")
    for _, r in err_pairs_head.head(3).iterrows():
        # truncate message for compactness
        msg = str(r['message'])
        if len(msg) > 120:
            msg = msg[:117] + "..."
        lines.append(f"  - {r['cmdb_id']} | count={int(r['count'])} | message_snippet=\"{msg}\"")
    if err_earliest5.shape[0] > 0:
        lines.append("- Earliest 5 timestamps in error_logs.csv (local tz Asia/Shanghai):")
        for ts in err_earliest5:
            lines.append(f"  - {ts}")
else:
    lines.append("- No error log pairs found.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows       min_timestamp_local       max_timestamp_local
0      metric.csv  26418 2021-03-24 11:30:00+08:00 2021-03-24 12:00:00+08:00
1       trace.csv   7044 2021-03-24 11:30:00+08:00 2021-03-24 12:00:00+08:00
2         log.csv    511 2021-03-24 11:30:00+08:00 2021-03-24 12:00:00+08:00
3  error_logs.csv    267 2021-03-24 11:31:01+08:00 2021-03-24 11:35:10+08:00,    cmdb_id                                           kpi_name  count  global_P95_of_value
0     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31         2.040136e+09
1     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage     19         4.351768e+01
2     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31         8.855733e+08
3     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31         5.850979e+07
4     IG01          JVM-Operating System_7778_JVM_JVM_CPULoad     17         5.178000e-01
..     ...                                                ...    ...                  ...
45    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...      9         1.800000e-01
46    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps      9         0.000000e+00
47    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead      9         0.000000e+00
48    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...     10         1.355000e+01
49    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps     11         1.000000e+00

[50 rows x 4 columns],    cmdb_id                         trace_name  count  global_P95_of_value
0     IG01           trace.self.duration_mean     31             6.984894
1     IG01            trace.self.duration_p95     31            60.302825
2     IG01               trace.self.row_count     31          1900.000000
3     IG01    trace.to_Tomcat01.duration_mean     30             0.868630
4     IG01     trace.to_Tomcat01.duration_p95     30             3.268802
..     ...                                ...    ...                  ...
45    MG01  trace.from_dockerA2.duration_mean     31             0.403975
46    MG01   trace.from_dockerA2.duration_p95     31             1.510350
47    MG01      trace.from_dockerA2.row_count     31            87.500000
48    MG01  trace.from_dockerB1.duration_mean     30             0.379452
49    MG01   trace.from_dockerB1.duration_p95     30             1.444842

[50 rows x 4 columns],      cmdb_id          log_name  count  global_P95_of_value
0    Mysql01   log.error_count     28                0.000
1    Mysql01     log.row_count     28               17.950
2    Mysql02   log.error_count      2                0.000
3    Mysql02     log.row_count      2                1.000
4   Tomcat01   log.error_count     31                0.000
5   Tomcat01  log.gc_pause_max     18                0.100
6   Tomcat01     log.row_count     31              480.500
7   Tomcat02   log.error_count     31                0.000
8   Tomcat02  log.gc_pause_max     20                0.090
9   Tomcat02     log.row_count     31              506.000
10  Tomcat03   log.error_count     31                0.000
11  Tomcat03  log.gc_pause_max     20                0.081
12  Tomcat03     log.row_count     31              495.000
13  Tomcat04   log.error_count     31                0.000
14  Tomcat04  log.gc_pause_max     21                0.090
15  Tomcat04     log.row_count     31              476.000
16  apache01   log.error_count     31               32.500
17  apache01     log.row_count     31              952.000
18  apache02   log.error_count     31               30.000
19  apache02     log.row_count     31              973.000,      cmdb_id                                            message  count
0   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
1   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
2   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
3   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
4   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
5   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
6   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
7   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
8   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
9   apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
10  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
11  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
12  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
13  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
14  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
15  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
16  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1
17  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1
18  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1
19  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1, 0   2021-03-24 11:31:01+08:00
1   2021-03-24 11:31:02+08:00
2   2021-03-24 11:31:04+08:00
3   2021-03-24 11:31:05+08:00
4   2021-03-24 11:31:07+08:00
Name: timestamp, dtype: datetime64[ns, Asia/Shanghai])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric from kernel. Ensure ts_local exists; if not, create it.
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
if 'ts_local' not in df_metric.columns:
    df_metric['ts_local'] = pd.to_datetime(df_metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire metric.csv
metric_global_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_P95_of_value=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.notna().any() else np.nan)
    .reset_index()
)

# 2) Filter df_metric to rows within 2021-03-24 11:30:00+08:00 through 2021-03-24 12:00:00+08:00 (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-24 11:30:00'))
end_local = tz.localize(pd.Timestamp('2021-03-24 12:00:00'))
metric_filtered = df_metric[
    (df_metric['ts_local'] >= start_local) &
    (df_metric['ts_local'] <= end_local)
].copy()

# 3) Join the filtered rows with metric_global_p95 on (cmdb_id, kpi_name)
metric_joined = metric_filtered.merge(metric_global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Detect anomalies where value >= global_P95_of_value
# Exclude rows where global_P95_of_value is NaN to avoid ambiguous comparisons
metric_joined['is_anom'] = False
mask_valid_p95 = metric_joined['global_P95_of_value'].notna()
metric_joined.loc[mask_valid_p95, 'is_anom'] = metric_joined.loc[mask_valid_p95, 'value'] >= metric_joined.loc[mask_valid_p95, 'global_P95_of_value']

metric_anoms = metric_joined[metric_joined['is_anom']].copy()

# 5) Aggregate anomalies by (cmdb_id, kpi_name)
if not metric_anoms.empty:
    agg = metric_anoms.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_timestamp_local=('ts_local', 'min'),
        max_value=('value', 'max'),
        global_P95_of_value=('global_P95_of_value', 'first')  # same per group
    ).reset_index()
    # compute severity_ratio = (max_value - P95)/P95, None if P95 == 0
    def compute_sev(row):
        p = row['global_P95_of_value']
        if p == 0 or pd.isna(p):
            return None
        return (row['max_value'] - p) / p
    agg['severity_ratio'] = agg.apply(compute_sev, axis=1)
    # order by anomaly_count desc, then severity_ratio desc (treat None as lowest)
    agg_sorted = agg.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False], na_position='last')
    # limit to top 20 rows
    metric_anoms_filtered = agg_sorted.head(20).reset_index(drop=True)
else:
    metric_anoms_filtered = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local',
        'max_value','global_P95_of_value','severity_ratio'
    ])

# 6) total number of anomaly points across all metrics
total_anoms_count = int(metric_anoms.shape[0])

# Keep outputs compact: show metric_global_p95 small sample, metric_anoms_filtered top rows, and total anomaly count
metric_global_p95_head = metric_global_p95.head(10)
metric_anoms_filtered, metric_global_p95_head, total_anoms_count

```
Out[2]:
```
# Summarize results in plain English using existing kernel variables.
# Display a concise summary string only.

top_n = 5
if 'metric_anoms_filtered' in globals() and not metric_anoms_filtered.empty:
    top_rows = metric_anoms_filtered.head(top_n)
    top_lines = []
    for _, r in top_rows.iterrows():
        ts = r['earliest_anomaly_timestamp_local']
        ts_str = ts.isoformat() if not pd.isna(ts) else "N/A"
        p95 = r['global_P95_of_value']
        maxv = r['max_value']
        sev = r['severity_ratio']
        sev_str = f"{sev:.6g}" if pd.notna(sev) and sev is not None else "None"
        top_lines.append(f"{r['cmdb_id']} | {r['kpi_name']} | count={int(r['anomaly_count'])} | earliest={ts_str} | max={maxv:.6g} | P95={p95:.6g} | severity_ratio={sev_str}")
else:
    top_lines = ["(no anomalies found)"]

total = globals().get('total_anoms_count', None)

summary_lines = [
    f"Total anomaly points detected across metric.csv (value >= global P95): {total}",
    "Top anomaly groups (up to 5):"
] + top_lines + [
    "",
    "Notes: An anomaly was defined as a metric point with value >= the global P95 computed across the entire metric.csv.",
    "Many top groups show severity_ratio = 0.0, which means the observed max equals the global P95 (i.e., points reach but do not exceed P95).",
    "Full anomaly table is stored in the kernel variable: metric_anoms_filtered"
]

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp_local     max_value  global_P95_of_value  severity_ratio
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31        2021-03-24 11:30:00+08:00  2.040136e+09         2.040136e+09             0.0
1       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31        2021-03-24 11:30:00+08:00  2.040136e+09         2.040136e+09             0.0
2       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        2021-03-24 11:30:00+08:00  2.040136e+09         2.040136e+09             0.0
3       MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        2021-03-24 11:30:00+08:00  2.000000e+00         2.000000e+00             0.0
4       MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        2021-03-24 11:30:00+08:00  3.000000e+00         3.000000e+00             0.0
5       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        2021-03-24 11:30:00+08:00  2.040136e+09         2.040136e+09             0.0
6       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31        2021-03-24 11:30:00+08:00  5.864708e+07         5.864708e+07             0.0
7       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        2021-03-24 11:30:00+08:00  1.000000e+00         1.000000e+00             0.0
8       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31        2021-03-24 11:30:00+08:00  3.000000e-01         3.000000e-01             0.0
9       MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        2021-03-24 11:30:00+08:00  3.000000e+00         3.000000e+00             0.0
10   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        2021-03-24 11:30:00+08:00  4.000000e+00         4.000000e+00             0.0
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31        2021-03-24 11:30:00+08:00  5.810000e+02         5.810000e+02             0.0
12  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31        2021-03-24 11:30:00+08:00  3.978080e+05         3.978080e+05             0.0
13  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31        2021-03-24 11:30:00+08:00  1.000000e+02         1.000000e+02             0.0
14  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31        2021-03-24 11:30:00+08:00  5.000000e+02         5.000000e+02             0.0
15  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31        2021-03-24 11:30:00+08:00  5.260000e+02         5.260000e+02             0.0
16  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31        2021-03-24 11:30:00+08:00  3.906190e+05         3.906190e+05             0.0
17  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31        2021-03-24 11:30:00+08:00  1.000000e+02         1.000000e+02             0.0
18  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31        2021-03-24 11:30:00+08:00  5.000000e+02         5.000000e+02             0.0
19  Tomcat03      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31        2021-03-24 11:30:00+08:00  1.000000e+00         1.000000e+00             0.0,   cmdb_id                                     kpi_name  global_P95_of_value
0    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax         2.040136e+09
1    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage         4.351768e+01
2    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed         8.855733e+08
3    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed         5.850979e+07
4    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad         5.178000e-01
5    IG01              JVM-Runtime_7778_JVM_JVM_Uptime         1.637277e+06
6    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads         7.600000e+01
7    IG01              OSLinux-CPU_CPU-0_SingleCpuUtil         1.993740e+00
8    IG01              OSLinux-CPU_CPU-0_SingleCpuidle         9.880844e+01
9    IG01              OSLinux-CPU_CPU-2_SingleCpuUtil         1.572125e+00, 15145)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_anoms_filtered from kernel
if 'metric_anoms_filtered' not in globals():
    metric_anoms_filtered = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local',
        'max_value','global_P95_of_value','severity_ratio'
    ])

# Ensure severity_ratio is numeric and earliest timestamp is timezone-aware datetime
metric_anoms_filtered['severity_ratio'] = pd.to_numeric(metric_anoms_filtered['severity_ratio'], errors='coerce')
# If earliest_anomaly_timestamp_local is not datetime, attempt conversion (should already be)
if metric_anoms_filtered.shape[0] > 0 and not pd.api.types.is_datetime64tz_dtype(metric_anoms_filtered['earliest_anomaly_timestamp_local']):
    metric_anoms_filtered['earliest_anomaly_timestamp_local'] = pd.to_datetime(metric_anoms_filtered['earliest_anomaly_timestamp_local'], utc=True).dt.tz_convert(tz)

# 1) Select severity_ratio > 0.5, fallback to > 0.1 if none match
mask_high = metric_anoms_filtered['severity_ratio'] > 0.5
if mask_high.any():
    selected = metric_anoms_filtered[mask_high].copy()
else:
    mask_fallback = metric_anoms_filtered['severity_ratio'] > 0.1
    selected = metric_anoms_filtered[mask_fallback].copy()

# 2) Order by severity_ratio desc and return up to top 20 rows with specified columns
cols_out = ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local','max_value','global_P95_of_value','severity_ratio']
if not selected.empty:
    selected_sorted = selected.sort_values(by='severity_ratio', ascending=False, na_position='last').reset_index(drop=True)
    metric_severe_anoms = selected_sorted[cols_out].head(20).reset_index(drop=True)
else:
    metric_severe_anoms = pd.DataFrame(columns=cols_out)

# 3) Aggregation of counts of these severe anomaly groups per cmdb_id
if metric_severe_anoms.shape[0] > 0:
    severe_counts_df = (
        metric_severe_anoms
        .groupby('cmdb_id', dropna=False)
        .agg(severe_kpi_count=('kpi_name', 'nunique'))
        .reset_index()
        .sort_values(by='severe_kpi_count', ascending=False)
        .reset_index(drop=True)
    )
else:
    severe_counts_df = pd.DataFrame(columns=['cmdb_id','severe_kpi_count'])

# 4) Global earliest_anomaly_timestamp_local among selected severe groups
if metric_severe_anoms.shape[0] > 0:
    global_earliest_ts = metric_severe_anoms['earliest_anomaly_timestamp_local'].min()
else:
    global_earliest_ts = pd.NaT

# Keep outputs compact: show metric_severe_anoms (up to 20 rows), severe_counts_df, and global_earliest_ts
metric_severe_anoms, severe_counts_df, global_earliest_ts

```
Out[3]:
```
summary = (
    f"Selection result: 0 severe anomaly groups found.\n"
    f"No groups met severity_ratio > 0.5, and fallback threshold > 0.1 also returned no rows.\n"
    f"Total anomaly groups available for consideration: {metric_anoms_filtered.shape[0]}.\n"
    f"Total anomaly points previously detected across metric.csv: {globals().get('total_anoms_count', 'unknown')}.\n"
    f"The selected severe anomalies table (kernel variable 'metric_severe_anoms') is empty.\n"
    f"Global earliest anomaly timestamp among selected severe groups: None."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, anomaly_count, earliest_anomaly_timestamp_local, max_value, global_P95_of_value, severity_ratio]
Index: [], Empty DataFrame
Columns: [cmdb_id, severe_kpi_count]
Index: [], NaT)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_err if available, otherwise load
if 'df_err' in globals():
    df = df_err.copy()
else:
    df = pd.read_csv('error_logs.csv')

# Ensure timestamp column exists and create tz-aware datetime, then floor to minute in Asia/Shanghai
df['ts_utc'] = pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True)
df['timestamp_minute_local'] = df['ts_utc'].dt.tz_convert(tz).dt.floor('T')

# 1) Aggregate counts per (cmdb_id, timestamp_minute_local)
error_spike_agg = (
    df
    .groupby(['cmdb_id', 'timestamp_minute_local'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)

# Keep only top 20 for compact display later, but store full table in error_spike_agg variable
error_spike_agg_head = error_spike_agg.head(20)

# 2) Identify cmdb_id with highest total error count across the file
total_by_cmdb = df.groupby('cmdb_id', dropna=False).size().reset_index(name='total_errors').sort_values(by='total_errors', ascending=False)
top_err_cmdb = total_by_cmdb.iloc[0]['cmdb_id'] if not total_by_cmdb.empty else None

# 3) For top_err_cmdb, get up to 10 distinct (message_snippet, count, earliest_timestamp_local)
if top_err_cmdb is not None:
    df_top = df[df['cmdb_id'] == top_err_cmdb].copy()
    # Ensure message string and snippet truncated to 200 chars
    df_top['message_str'] = df_top['message'].fillna('').astype(str)
    # Group by full message (to count identical messages), but present truncated snippet for compactness
    msg_agg = (
        df_top
        .groupby('message_str', dropna=False)
        .agg(
            count=('message_str', 'count'),
            earliest_ts_utc=('ts_utc', 'min')
        )
        .reset_index()
        .sort_values(by='count', ascending=False)
    )
    # Convert earliest timestamp to local tz and create snippet
    msg_agg['earliest_timestamp_local'] = msg_agg['earliest_ts_utc'].dt.tz_convert(tz)
    msg_agg['message_snippet'] = msg_agg['message_str'].str.slice(0, 200)
    error_top_msgs = msg_agg[['message_snippet', 'count', 'earliest_timestamp_local']].head(10).reset_index(drop=True)
else:
    error_top_msgs = pd.DataFrame(columns=['message_snippet','count','earliest_timestamp_local'])

# Compact outputs: top 20 spike aggregates, top_err_cmdb identifier, and up to 10 top messages for that cmdb
error_spike_agg_head, top_err_cmdb, error_top_msgs

```
Out[4]:
```
summary_lines = []

# Top spikes (use error_spike_agg if available)
if 'error_spike_agg' in globals() and not error_spike_agg.empty:
    top_spikes = error_spike_agg.head(3)
    summary_lines.append("Top error spikes (top 3):")
    for _, r in top_spikes.iterrows():
        ts = r['timestamp_minute_local']
        ts_str = ts.isoformat() if hasattr(ts, 'isoformat') else str(ts)
        summary_lines.append(f"- {r['cmdb_id']} at {ts_str}: count={int(r['count'])}")
else:
    summary_lines.append("No error spike aggregates available.")

# Top error host
top_err = globals().get('top_err_cmdb', None)
if top_err:
    summary_lines.append(f"\nHost with highest total errors: {top_err}")
else:
    summary_lines.append("\nNo top error host identified.")

# Top messages for that host
if 'error_top_msgs' in globals() and not error_top_msgs.empty:
    summary_lines.append(f"\nTop messages for {top_err} (up to 10; snippets truncated to 200 chars):")
    for _, r in error_top_msgs.iterrows():
        ts = r['earliest_timestamp_local']
        ts_str = ts.isoformat() if hasattr(ts, 'isoformat') else str(ts)
        msg = r['message_snippet'].replace("\n", " ")
        summary_lines.append(f"- count={int(r['count'])} | earliest={ts_str} | msg=\"{msg}\"")
    # earliest among these
    earliest = error_top_msgs['earliest_timestamp_local'].min()
    earliest_str = earliest.isoformat() if hasattr(earliest, 'isoformat') and pd.notna(earliest) else "N/A"
    summary_lines.append(f"\nEarliest occurrence among these messages: {earliest_str}")
else:
    summary_lines.append(f"\nNo top messages available for {top_err}.")

summary_lines.append("\nNotes: Full aggregates are stored in kernel variables 'error_spike_agg' and 'error_top_msgs'.")

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id    timestamp_minute_local  count
0  apache01 2021-03-24 11:31:00+08:00     52
1  apache01 2021-03-24 11:34:00+08:00     34
2  apache02 2021-03-24 11:31:00+08:00     33
3  apache02 2021-03-24 11:34:00+08:00     32
4  apache01 2021-03-24 11:33:00+08:00     31
5  apache02 2021-03-24 11:32:00+08:00     28
6  apache01 2021-03-24 11:32:00+08:00     24
7  apache02 2021-03-24 11:33:00+08:00     24
8  apache02 2021-03-24 11:35:00+08:00      7
9  apache01 2021-03-24 11:35:00+08:00      2, 'apache01',                                      message_snippet  count  earliest_timestamp_local
0  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:05+08:00
1  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:34:33+08:00
2  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:34:45+08:00
3  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:34:06+08:00
4  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:34:56+08:00
5  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:39+08:00
6  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:33:12+08:00
7  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:12+08:00
8  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:21+08:00
9  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:34:35+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace if present, otherwise load
if 'df_trace' in globals():
    df = df_trace.copy()
else:
    df = pd.read_csv('trace.csv')
# Ensure ts_local exists as Asia/Shanghai tz-aware datetime
if 'ts_local' not in df.columns or not pd.api.types.is_datetime64tz_dtype(df['ts_local']):
    df['ts_local'] = pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using entire trace.csv
trace_global_thresholds = (
    df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        global_P95_of_value=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.notna().any() else np.nan,
        global_P05_of_value=lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.notna().any() else np.nan
    )
    .reset_index()
)

# 2) Filter to incident window in local tz (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-24 11:30:00'))
end_local = tz.localize(pd.Timestamp('2021-03-24 12:00:00'))
df_filtered = df[(df['ts_local'] >= start_local) & (df['ts_local'] <= end_local)].copy()

# 3) Join filtered rows with trace_global_thresholds
df_joined = df_filtered.merge(trace_global_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# 4) Mark anomalies:
# - if trace_name contains 'row_count' (case-insensitive) => low anomaly when value <= P05
# - otherwise => high anomaly when value >= P95
df_joined['is_row_count'] = df_joined['trace_name'].str.contains('row_count', case=False, na=False)
# Only consider rows where thresholds are available (not NaN)
df_joined['is_anom'] = False
mask_valid = df_joined['global_P95_of_value'].notna() | df_joined['global_P05_of_value'].notna()
# high anomalies
mask_high = mask_valid & (~df_joined['is_row_count']) & df_joined['global_P95_of_value'].notna() & (df_joined['value'] >= df_joined['global_P95_of_value'])
# low anomalies
mask_low = mask_valid & (df_joined['is_row_count']) & df_joined['global_P05_of_value'].notna() & (df_joined['value'] <= df_joined['global_P05_of_value'])
df_joined.loc[mask_high, 'is_anom'] = True
df_joined.loc[mask_low, 'is_anom'] = True
df_joined.loc[mask_high, 'anom_type'] = 'high'
df_joined.loc[mask_low, 'anom_type'] = 'low'

trace_anom_points = df_joined[df_joined['is_anom']].copy()

# 5) Aggregate anomalies by (cmdb_id, trace_name)
if not trace_anom_points.empty:
    agg = trace_anom_points.groupby(['cmdb_id', 'trace_name', 'anom_type'], dropna=False).agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_timestamp_local=('ts_local', 'min'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        global_P95_of_value=('global_P95_of_value', 'first'),
        global_P05_of_value=('global_P05_of_value', 'first')
    ).reset_index()

    # compute severity_ratio depending on anom_type
    def compute_sev(row):
        try:
            if row['anom_type'] == 'high':
                p95 = row['global_P95_of_value']
                if pd.isna(p95) or p95 == 0:
                    return None
                return (row['max_value'] - p95) / p95
            else:  # low
                p05 = row['global_P05_of_value']
                if pd.isna(p05) or p05 == 0:
                    return None
                return (p05 - row['min_value']) / p05
        except Exception:
            return None

    agg['severity_ratio'] = agg.apply(compute_sev, axis=1)

    # Order by severity_ratio desc (NaNs last), then anomaly_count desc
    agg_sorted = agg.sort_values(by=['severity_ratio', 'anomaly_count'], ascending=[False, False], na_position='last').reset_index(drop=True)

    # Keep required columns and limit to top 20
    trace_anoms_filtered = agg_sorted[[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp_local',
        'min_value','max_value','global_P95_of_value','global_P05_of_value','severity_ratio'
    ]].head(20).reset_index(drop=True)
else:
    trace_anoms_filtered = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp_local',
        'min_value','max_value','global_P95_of_value','global_P05_of_value','severity_ratio'
    ])

# 6) total number of trace anomaly points detected
total_trace_anoms_count = int(trace_anom_points.shape[0])

# Keep outputs compact: show top 20 aggregated anomalies and total count; also keep full thresholds in variable trace_global_thresholds
trace_anoms_filtered, total_trace_anoms_count, trace_global_thresholds.head(10)

```
Out[5]:
```
# Summarize the trace anomaly analysis in plain English using existing kernel variables.
# The summary will reference: trace_anoms_filtered, total_trace_anoms_count, trace_global_thresholds

import pandas as pd

# Prepare top rows for summary (up to 5 for compactness)
top = trace_anoms_filtered.head(5) if 'trace_anoms_filtered' in globals() else pd.DataFrame()

lines = []
lines.append("Trace anomaly analysis summary:")
lines.append("")
# thresholds presence
if 'trace_global_thresholds' in globals():
    lines.append("- Global P95/P05 thresholds were computed across the full trace.csv (stored in variable 'trace_global_thresholds').")
else:
    lines.append("- Global thresholds not found in kernel variables.")

# total anomalies
total = globals().get('total_trace_anoms_count', None)
lines.append(f"- Total trace anomaly points detected in the incident window: {total}")
lines.append("- Anomalies are defined as: for '*row_count*' traces -> low anomalies (value <= global P05); otherwise -> high anomalies (value >= global P95).")
lines.append("")

# Top anomaly groups
if not top.empty:
    lines.append("Top anomaly groups (up to 5):")
    for _, r in top.iterrows():
        cmdb = r['cmdb_id']
        name = r['trace_name']
        cnt = int(r['anomaly_count'])
        ts = r['earliest_anomaly_timestamp_local']
        ts_str = ts.isoformat() if hasattr(ts, 'isoformat') and pd.notna(ts) else "N/A"
        minv = r['min_value']
        maxv = r['max_value']
        p95 = r['global_P95_of_value']
        p05 = r['global_P05_of_value']
        sev = r['severity_ratio']
        sev_str = f"{sev:.6g}" if pd.notna(sev) and sev is not None else "None"
        lines.append(f"- {cmdb} | {name} | count={cnt} | earliest={ts_str} | min={minv:.6g} | max={maxv:.6g} | P95={p95:.6g} | P05={p05:.6g} | severity_ratio={sev_str}")
    # highlight highest severity
    highest = top.iloc[0]
    lines.append("")
    lines.append(f"- Highest severity_ratio in the returned list: {highest['severity_ratio']:.6g} (example: {highest['cmdb_id']} | {highest['trace_name']}).")
else:
    lines.append("No trace anomaly groups found in the incident window.")

lines.append("")
lines.append("Notes:")
lines.append("- Results are stored in kernel variables: 'trace_anoms_filtered' (aggregated anomalies) and 'trace_global_thresholds' (global P95/P05).")
lines.append("- severity_ratio is calculated as: for high anomalies (max - P95)/P95; for low anomalies (P05 - min)/P05. If P95 or P05 == 0 or missing, severity_ratio is None.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  anomaly_count earliest_anomaly_timestamp_local  min_value  max_value  global_P95_of_value  global_P05_of_value  severity_ratio
0       MG01   trace.from_dockerB1.duration_p95              2        2021-03-24 11:30:00+08:00   1.784750   5.583000             1.444842             0.035955        2.864089
1   dockerB1         trace.to_MG01.duration_p95              2        2021-03-24 11:30:00+08:00   1.784750   5.583000             1.444842             0.035955        2.864089
2       MG01  trace.from_dockerB2.duration_mean              2        2021-03-24 11:30:00+08:00   0.211674   0.658534             0.208494             0.033025        2.158523
3   dockerB2        trace.to_MG01.duration_mean              2        2021-03-24 11:30:00+08:00   0.211674   0.658534             0.208494             0.033025        2.158523
4       MG01   trace.from_dockerA2.duration_p95              2        2021-03-24 11:30:00+08:00   2.005700   4.660400             1.510350             0.026600        2.085642
5   dockerA2         trace.to_MG01.duration_p95              2        2021-03-24 11:30:00+08:00   2.005700   4.660400             1.510350             0.026600        2.085642
6       MG02   trace.from_dockerB1.duration_p95              2        2021-03-24 11:30:00+08:00   1.593500   3.932000             1.308250             0.028950        2.005542
7   dockerB1         trace.to_MG02.duration_p95              2        2021-03-24 11:30:00+08:00   1.593500   3.932000             1.308250             0.028950        2.005542
8       MG02     trace.to_dockerA1.duration_p95              2        2021-03-24 11:33:00+08:00   1.891000   4.202150             1.722375             0.343800        1.439742
9   dockerA1       trace.from_MG02.duration_p95              2        2021-03-24 11:33:00+08:00   1.891000   4.202150             1.722375             0.343800        1.439742
10      MG01   trace.from_dockerB2.duration_p95              2        2021-03-24 11:30:00+08:00   1.487400   2.939200             1.293000             0.052818        1.273163
11  dockerB2         trace.to_MG01.duration_p95              2        2021-03-24 11:30:00+08:00   1.487400   2.939200             1.293000             0.052818        1.273163
12      IG01     trace.to_Tomcat01.duration_p95              2        2021-03-24 11:40:00+08:00   3.403150   7.427700             3.268802             1.049573        1.272300
13  Tomcat01       trace.from_IG01.duration_p95              2        2021-03-24 11:40:00+08:00   3.403150   7.427700             3.268802             1.049573        1.272300
14      MG02  trace.from_dockerB2.duration_mean              2        2021-03-24 11:30:00+08:00   0.209202   0.424637             0.203222             0.020915        1.089523
15  dockerB2        trace.to_MG02.duration_mean              2        2021-03-24 11:30:00+08:00   0.209202   0.424637             0.203222             0.020915        1.089523
16  dockerA1            trace.self.duration_p95              2        2021-03-24 11:30:00+08:00   0.086150   0.117000             0.058175             0.001000        1.011173
17      MG01  trace.from_dockerB1.duration_mean              2        2021-03-24 11:30:00+08:00   0.526063   0.759667             0.379452             0.036898        1.002010
18  dockerB1        trace.to_MG01.duration_mean              2        2021-03-24 11:30:00+08:00   0.526063   0.759667             0.379452             0.036898        1.002010
19      MG01           trace.self.duration_mean              2        2021-03-24 11:30:00+08:00   0.372957   0.741840             0.371262             0.102343        0.998158, 500,   cmdb_id                       trace_name  global_P95_of_value  global_P05_of_value
0    IG01         trace.self.duration_mean             6.984894             0.327246
1    IG01          trace.self.duration_p95            60.302825             1.262625
2    IG01             trace.self.row_count          1900.000000          1195.000000
3    IG01  trace.to_Tomcat01.duration_mean             0.868630             0.261330
4    IG01   trace.to_Tomcat01.duration_p95             3.268802             1.049573
5    IG01      trace.to_Tomcat01.row_count           237.950000            86.000000
6    IG01  trace.to_Tomcat02.duration_mean             0.973351             0.217846
7    IG01   trace.to_Tomcat02.duration_p95             4.710700             1.046250
8    IG01      trace.to_Tomcat02.row_count           249.500000           142.000000
9    IG01  trace.to_Tomcat03.duration_mean             0.833552             0.213809)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log if present; otherwise load
if 'df_log' in globals():
    df = df_log.copy()
else:
    df = pd.read_csv('log.csv')

# Ensure ts_local exists and is tz-aware in Asia/Shanghai
if 'ts_local' not in df.columns or not pd.api.types.is_datetime64tz_dtype(df['ts_local']):
    df['ts_local'] = pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, log_name) using the entire log.csv BEFORE filtering
log_global_p95 = (
    df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(global_P95_of_value=lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.notna().any() else np.nan)
    .reset_index()
)
# small sample for inspection (kept in memory)
log_global_p95_head = log_global_p95.head(10)

# 2) Filter to the incident window in Asia/Shanghai (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-24 11:30:00'))
end_local = tz.localize(pd.Timestamp('2021-03-24 12:00:00'))
df_filtered = df[(df['ts_local'] >= start_local) & (df['ts_local'] <= end_local)].copy()

# 3) Join filtered rows with global P95
df_joined = df_filtered.merge(log_global_p95, on=['cmdb_id', 'log_name'], how='left')

# 4) Detect anomalies where value >= global_P95_of_value (only when P95 is not NaN)
mask_valid_p95 = df_joined['global_P95_of_value'].notna()
df_joined['is_anom'] = False
df_joined.loc[mask_valid_p95, 'is_anom'] = df_joined.loc[mask_valid_p95, 'value'] >= df_joined.loc[mask_valid_p95, 'global_P95_of_value']

log_anom_points = df_joined[df_joined['is_anom']].copy()

# 5) Aggregate anomalies by (cmdb_id, log_name)
if not log_anom_points.empty:
    agg = log_anom_points.groupby(['cmdb_id', 'log_name'], dropna=False).agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_timestamp_local=('ts_local', 'min'),
        max_value=('value', 'max'),
        global_P95_of_value=('global_P95_of_value', 'first')
    ).reset_index()

    # severity_ratio = (max_value - P95)/P95, None if P95==0 or NaN
    def compute_sev(row):
        p = row['global_P95_of_value']
        if pd.isna(p) or p == 0:
            return None
        return (row['max_value'] - p) / p

    agg['severity_ratio'] = agg.apply(compute_sev, axis=1)

    # order by anomaly_count desc (as requested)
    agg_sorted = agg.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False], na_position='last').reset_index(drop=True)
    log_anoms_filtered = agg_sorted[['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp_local','max_value','global_P95_of_value','severity_ratio']].head(20).reset_index(drop=True)
else:
    log_anoms_filtered = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_anomaly_timestamp_local','max_value','global_P95_of_value','severity_ratio'])

# 6) total number of log anomaly points detected and per-cmdb summary
total_log_anoms_count = int(log_anom_points.shape[0])

if not log_anoms_filtered.empty:
    log_anom_counts = (
        log_anoms_filtered
        .groupby('cmdb_id', dropna=False)
        .agg(total_log_anom_count=('anomaly_count', 'sum'))
        .reset_index()
        .sort_values(by='total_log_anom_count', ascending=False)
        .reset_index(drop=True)
    )
else:
    # If no aggregated anomalies, still provide empty structure
    log_anom_counts = pd.DataFrame(columns=['cmdb_id','total_log_anom_count'])

# Compact outputs: aggregated anomalies (top 20), per-cmdb counts, total anomaly points, and a small sample of global P95
log_anoms_filtered, log_anom_counts, total_log_anoms_count, log_global_p95_head

```
Out[6]:
```
summary = (
    "Log anomaly summary:\n"
    "- Total log anomaly points detected in the incident window: 181.\n"
    "- Top affected hosts by total log anomaly groups: Tomcat02=35, Tomcat01=35, Tomcat04=35, Tomcat03=34, Mysql01=30 (others: Mysql02/apache01/apache02=4 each).\n"
    "- Notable anomaly groups (examples):\n"
    "  * Many Tomcat 'log.error_count' groups show anomaly_count=31 but have global P95=0 (max=0) so severity_ratio is undefined.\n"
    "  * Mysql01 'log.row_count' (anomaly_count=2): max=31, P95=17.95, severity_ratio≈0.727 — notable increase.\n"
    "  * apache01 'log.error_count' (anomaly_count=2): max=52, P95=32.5, severity_ratio=0.60 — notable spike.\n"
    "  * Some 'log.row_count' and 'log.gc_pause_max' entries on Tomcat hosts show small positive severity ratios (e.g., Tomcat03 gc_pause_max severity≈0.235).\n"
    "- Interpretation: Most Tomcat error_count series have P95=0 so hits at 0 were counted as anomalies (value >= P95). However, real spikes with sizable severity appear on Mysql01 (row_count) and apache01 (error_count).\n"
    "- Data stored in kernel variables: 'log_anoms_filtered' (aggregated anomalies) and 'log_anom_counts' (per-host totals)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  anomaly_count earliest_anomaly_timestamp_local  max_value  global_P95_of_value  severity_ratio
0   Tomcat01   log.error_count             31        2021-03-24 11:30:00+08:00       0.00                0.000             NaN
1   Tomcat02   log.error_count             31        2021-03-24 11:30:00+08:00       0.00                0.000             NaN
2   Tomcat03   log.error_count             31        2021-03-24 11:30:00+08:00       0.00                0.000             NaN
3   Tomcat04   log.error_count             31        2021-03-24 11:30:00+08:00       0.00                0.000             NaN
4    Mysql01   log.error_count             28        2021-03-24 11:30:00+08:00       0.00                0.000             NaN
5    Mysql01     log.row_count              2        2021-03-24 11:40:00+08:00      31.00               17.950        0.727019
6   apache01   log.error_count              2        2021-03-24 11:31:00+08:00      52.00               32.500        0.600000
7   Tomcat04     log.row_count              2        2021-03-24 11:39:00+08:00     525.00              476.000        0.102941
8   apache02   log.error_count              2        2021-03-24 11:31:00+08:00      33.00               30.000        0.100000
9   apache01     log.row_count              2        2021-03-24 11:39:00+08:00    1001.00              952.000        0.051471
10  apache02     log.row_count              2        2021-03-24 11:39:00+08:00    1021.00              973.000        0.049332
11  Tomcat03     log.row_count              2        2021-03-24 11:39:00+08:00     509.00              495.000        0.028283
12  Tomcat01     log.row_count              2        2021-03-24 11:36:00+08:00     483.00              480.500        0.005203
13  Tomcat02     log.row_count              2        2021-03-24 11:41:00+08:00     508.00              506.000        0.003953
14   Mysql02     log.row_count              2        2021-03-24 11:33:00+08:00       1.00                1.000        0.000000
15  Tomcat01  log.gc_pause_max              2        2021-03-24 11:38:00+08:00       0.10                0.100        0.000000
16  Tomcat02  log.gc_pause_max              2        2021-03-24 11:48:00+08:00       0.09                0.090        0.000000
17  Tomcat04  log.gc_pause_max              2        2021-03-24 11:30:00+08:00       0.09                0.090        0.000000
18   Mysql02   log.error_count              2        2021-03-24 11:33:00+08:00       0.00                0.000             NaN
19  Tomcat03  log.gc_pause_max              1        2021-03-24 11:41:00+08:00       0.10                0.081        0.234568,     cmdb_id  total_log_anom_count
0  Tomcat02                    35
1  Tomcat01                    35
2  Tomcat04                    35
3  Tomcat03                    34
4   Mysql01                    30
5   Mysql02                     4
6  apache01                     4
7  apache02                     4, 181,     cmdb_id          log_name  global_P95_of_value
0   Mysql01   log.error_count                 0.00
1   Mysql01     log.row_count                17.95
2   Mysql02   log.error_count                 0.00
3   Mysql02     log.row_count                 1.00
4  Tomcat01   log.error_count                 0.00
5  Tomcat01  log.gc_pause_max                 0.10
6  Tomcat01     log.row_count               480.50
7  Tomcat02   log.error_count                 0.00
8  Tomcat02  log.gc_pause_max                 0.09
9  Tomcat02     log.row_count               506.00)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_err if available, otherwise load
if 'df_err' in globals():
    df = df_err.copy()
else:
    df = pd.read_csv('error_logs.csv')

# Ensure ts_utc and ts_local exist and are tz-aware in Asia/Shanghai
df['ts_utc'] = pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True)
df['ts_local'] = df['ts_utc'].dt.tz_convert(tz)

# Define filter window (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-24 11:30:00'))
end_local = tz.localize(pd.Timestamp('2021-03-24 11:36:00'))

# Filter rows for apache01 within the window
apache01_error_logs_window = df[
    (df['cmdb_id'] == 'apache01') &
    (df['ts_local'] >= start_local) &
    (df['ts_local'] <= end_local)
].copy()

# Add minute-floor column for aggregation
apache01_error_logs_window['timestamp_minute_local'] = apache01_error_logs_window['ts_local'].dt.floor('T')

# 1) Aggregate distinct full message texts: message (full), count, earliest_timestamp_local
apache01_error_msg_counts = (
    apache01_error_logs_window
    .groupby('message', dropna=False)
    .agg(
        count=('message', 'count'),
        earliest_timestamp_local=('ts_local', 'min')
    )
    .reset_index()
    .sort_values(by=['count', 'earliest_timestamp_local'], ascending=[False, True])
    .head(20)
    .reset_index(drop=True)
)

# 2) Total number of apache01 error log rows in this interval
total_apache01_rows = int(apache01_error_logs_window.shape[0])

# 3) Exact minute with the highest count (timestamp_minute_local and count)
if not apache01_error_logs_window.empty:
    minute_agg = (
        apache01_error_logs_window
        .groupby('timestamp_minute_local', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values(by=['count', 'timestamp_minute_local'], ascending=[False, True])
        .reset_index(drop=True)
    )
    top_minute = minute_agg.head(1)
else:
    top_minute = pd.DataFrame(columns=['timestamp_minute_local', 'count'])

# Keep outputs compact: show the aggregated message counts (up to 20), total row count, and top minute
apache01_error_msg_counts, total_apache01_rows, top_minute

```
Out[7]:
```
summary_parts = []

# Use kernel variables produced earlier
total = globals().get('total_apache01_rows', None)
top_minute_df = globals().get('top_minute', None)
msg_df = globals().get('apache01_error_msg_counts', None)

summary_parts.append(f"Apache01 error logs in window 2021-03-24 11:30:00+08:00 through 2021-03-24 11:36:00+08:00:")

# Total rows
summary_parts.append(f"- Total rows in interval: {int(total) if total is not None else 'unknown'}")

# Peak minute
if isinstance(top_minute_df, (pd.DataFrame,)) and not top_minute_df.empty:
    ts = top_minute_df.iloc[0]['timestamp_minute_local']
    cnt = int(top_minute_df.iloc[0]['count'])
    summary_parts.append(f"- Peak minute: {ts} with count = {cnt}")
else:
    summary_parts.append("- Peak minute: N/A")

# Message aggregation summary
if isinstance(msg_df, (pd.DataFrame,)) and not msg_df.empty:
    summary_parts.append(f"- Distinct full messages aggregated (showing up to 20). Most messages in this window appear once.")
    # show up to 5 example messages for quick context
    examples = msg_df['message'].head(5).tolist()
    for i, m in enumerate(examples, 1):
        one_line = " ".join(str(m).splitlines())  # make single-line
        summary_parts.append(f"  {i}. {one_line} (count=1, earliest={msg_df.iloc[i-1]['earliest_timestamp_local']})")
else:
    summary_parts.append("- No aggregated messages found.")

summary_parts.append("")
summary_parts.append("Stored variables in kernel:")
summary_parts.append("- apache01_error_logs_window : full filtered rows for apache01 in the window")
summary_parts.append("- apache01_error_msg_counts : aggregated distinct messages with counts and earliest timestamps")

summary = "\n".join(summary_parts)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                              message  count  earliest_timestamp_local
0   IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:01+08:00
1   IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-24 11:31:02+08:00
2   IPAddress "POST /UOCP/base/ServiceTest4.json H...      1 2021-03-24 11:31:04+08:00
3   IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:04+08:00
4   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:04+08:00
5   IPAddress "POST /UOCP/base/ServiceTest8.json H...      1 2021-03-24 11:31:04+08:00
6   IPAddress "POST /UOCP/person/ServiceTest9.json...      1 2021-03-24 11:31:04+08:00
7   IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:05+08:00
8   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:07+08:00
9   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:09+08:00
10  IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:11+08:00
11  IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:11+08:00
12  IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-24 11:31:11+08:00
13  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:12+08:00
14  IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-24 11:31:12+08:00
15  IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:14+08:00
16  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1 2021-03-24 11:31:19+08:00
17  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1 2021-03-24 11:31:19+08:00
18  IPAddress "POST /UOCP/person/ServiceTest9.json...      1 2021-03-24 11:31:19+08:00
19  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:20+08:00, 143,      timestamp_minute_local  count
0 2021-03-24 11:31:00+08:00     52)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_err if present; otherwise load
if 'df_err' in globals():
    df = df_err.copy()
else:
    df = pd.read_csv('error_logs.csv')

# Ensure ts_utc and ts_local exist and are tz-aware in Asia/Shanghai
if 'ts_utc' not in df.columns:
    df['ts_utc'] = pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True)
df['ts_local'] = df['ts_utc'].dt.tz_convert(tz)

# Define window (inclusive)
start_local = tz.localize(pd.Timestamp('2021-03-24 11:30:00'))
end_local = tz.localize(pd.Timestamp('2021-03-24 11:36:00'))

# Filter rows for apache01 in window and store full filtered table
apache01_error_logs_window = df[
    (df['cmdb_id'] == 'apache01') &
    (df['ts_local'] >= start_local) &
    (df['ts_local'] <= end_local)
].copy()

# Add minute-floor column for aggregation
apache01_error_logs_window['timestamp_minute_local'] = apache01_error_logs_window['ts_local'].dt.floor('T')

# 1) Total row count and minute with highest count
total_apache01_rows = int(apache01_error_logs_window.shape[0])

if total_apache01_rows > 0:
    minute_agg = (
        apache01_error_logs_window
        .groupby('timestamp_minute_local', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values(by=['count', 'timestamp_minute_local'], ascending=[False, True])
        .reset_index(drop=True)
    )
    top_minute = minute_agg.head(1)
else:
    minute_agg = pd.DataFrame(columns=['timestamp_minute_local','count'])
    top_minute = pd.DataFrame(columns=['timestamp_minute_local','count'])

# 2) Extract HTTP status code per distinct message
# Prepare message text
apache01_error_logs_window['message_str'] = apache01_error_logs_window['message'].fillna('').astype(str)

# Define extraction function with prioritized patterns
status_patterns = [
    r'status[:=]\s*(\d{3})',                    # status=200 or status: 500
    r'HTTP/\d\.\d"\s*(\d{3})',                  # HTTP/1.1" 500
    r'"\s*(\d{3})\s',                           # "... " 500 <maybe>
    r'\b([45]\d{2})\b'                          # standalone 4xx/5xx
]

def extract_status(msg):
    for pat in status_patterns:
        m = re.search(pat, msg)
        if m:
            return m.group(1)
    return 'UNKNOWN'

# Apply extraction on unique messages to avoid repeated work
unique_msgs = apache01_error_logs_window[['message_str','ts_local']].copy()
unique_msgs['status_code'] = unique_msgs['message_str'].apply(extract_status)

# 3) Aggregate by extracted status code: count, earliest_timestamp_local, example_message_snippet
status_agg = (
    unique_msgs
    .groupby('status_code', dropna=False)
    .agg(
        count=('message_str', 'count'),
        earliest_timestamp_local=('ts_local', 'min')
    )
    .reset_index()
)

# For example_message_snippet, pick one message per status_code (earliest) and truncate to 300 chars
# Merge to get an example message
example_msgs = (
    unique_msgs
    .sort_values(by=['status_code', 'ts_local'], ascending=[True, True])
    .drop_duplicates(subset=['status_code'], keep='first')
    .loc[:, ['status_code', 'message_str']]
)
example_msgs['example_message_snippet'] = example_msgs['message_str'].str.slice(0, 300)

apache01_status_counts = status_agg.merge(example_msgs[['status_code','example_message_snippet']], on='status_code', how='left')
apache01_status_counts = apache01_status_counts.sort_values(by='count', ascending=False).reset_index(drop=True).head(20)

# 4) Distinct full messages with counts and earliest timestamps (store up to 20 for compact display)
apache01_error_msg_counts = (
    apache01_error_logs_window
    .groupby('message_str', dropna=False)
    .agg(
        count=('message_str', 'count'),
        earliest_timestamp_local=('ts_local', 'min')
    )
    .reset_index()
    .rename(columns={'message_str': 'message'})
    .sort_values(by=['count','earliest_timestamp_local'], ascending=[False, True])
    .reset_index(drop=True)
    .head(20)
)

# Final outputs (variables to keep in kernel). Display compact summaries as variables.
apache01_error_msg_counts, apache01_status_counts, total_apache01_rows, top_minute

```
Out[8]:
```
summary = (
    "apache01 error-log window (2021-03-24 11:30:00+08:00 through 2021-03-24 11:36:00+08:00):\n"
    "- Total rows in interval: 143\n"
    "- Peak minute: 2021-03-24 11:31:00+08:00 with 52 rows\n"
    "- HTTP status extraction: most messages map to status code '502' (143 occurrences). Earliest 502 observed: 2021-03-24 11:31:01+08:00.\n"
    "  Example message snippet (truncated): \"IPAddress \\\"POST /UOCP/base/ServiceTest6.json H...\"\n"
    "- Distinct full messages: up to 20 distinct messages were aggregated (most occur once). See variable 'apache01_error_msg_counts' for full aggregated list with earliest timestamps.\n"
    "- Kernel variables saved:\n"
    "    * apache01_error_logs_window  -> full filtered rows for apache01 in the window\n"
    "    * apache01_status_counts      -> aggregated counts by extracted status_code (shows 502=143)\n"
    "    * apache01_error_msg_counts   -> distinct full messages with counts and earliest timestamps\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                              message  count  earliest_timestamp_local
0   IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:01+08:00
1   IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-24 11:31:02+08:00
2   IPAddress "POST /UOCP/base/ServiceTest4.json H...      1 2021-03-24 11:31:04+08:00
3   IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:04+08:00
4   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:04+08:00
5   IPAddress "POST /UOCP/base/ServiceTest8.json H...      1 2021-03-24 11:31:04+08:00
6   IPAddress "POST /UOCP/person/ServiceTest9.json...      1 2021-03-24 11:31:04+08:00
7   IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:05+08:00
8   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:07+08:00
9   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:09+08:00
10  IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:11+08:00
11  IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-24 11:31:11+08:00
12  IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-24 11:31:11+08:00
13  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:12+08:00
14  IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-24 11:31:12+08:00
15  IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-24 11:31:14+08:00
16  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1 2021-03-24 11:31:19+08:00
17  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1 2021-03-24 11:31:19+08:00
18  IPAddress "POST /UOCP/person/ServiceTest9.json...      1 2021-03-24 11:31:19+08:00
19  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-24 11:31:20+08:00,   status_code  count  earliest_timestamp_local                            example_message_snippet
0         502    143 2021-03-24 11:31:01+08:00  IPAddress "POST /UOCP/base/ServiceTest6.json H..., 143,      timestamp_minute_local  count
0 2021-03-24 11:31:00+08:00     52)```
```